# NLP + Transformers


---
## 1) Visualización de la Configuración de BERT 👀

In [4]:
#!pip install transformers

In [5]:
from transformers import BertConfig, BertModel

Cargar la configuración del modelo BERT en español

In [6]:
config = BertConfig.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
print("Número de capas de BERT:", config.num_hidden_layers)

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

Número de capas de BERT: 12


Cargar el modelo para visualizar su arquitectura

In [7]:
model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", output_hidden_states=True)
print(model)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31002, 768, padding_idx=1)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

### Análisis Técnico — Sección 1

1. **Módulos principales de la arquitectura BERT (`BertModel`):**
   * **`embeddings` (`BertEmbeddings`):** Convierte los identificadores numéricos de entrada (tokens, segmentos y posiciones) en vectores densos continuos. Está compuesto por `word_embeddings` (vocabulario de 31,002 tokens en BETO), `position_embeddings` (hasta 512 posiciones) y `token_type_embeddings` (2 tipos de segmento), seguidos de normalización (`LayerNorm`) y `Dropout`.
   * **`encoder` (`BertEncoder`):** Es el núcleo de procesamiento contextual, conformado por una pila secuencial de **12 bloques codificadores idénticos (`BertLayer`)** numerados del `0` al `11`. Cada capa refina las representaciones vectoriales aplicando atención bidireccional sobre toda la secuencia.
   * **`pooler` (`BertPooler`):** Toma exclusivamente el vector oculto correspondiente al primer token de la secuencia (`[CLS]`) de la última capa del encoder, lo pasa por una capa lineal densa (`768 -> 768`) y aplica una función de activación `Tanh()` para generar una representación fija de toda la oración (útil en tareas de clasificación).

2. **Subcapas internas de un bloque `BertLayer`:**
   * **Mecanismo de Atención (`BertSelfAttention`):** Contiene las tres proyecciones lineales fundamentales: `query` ($Q$), `key` ($K$) y `value` ($V$), cada una mapeando de `768` a `768` dimensiones (divididas internamente en 12 cabezas de atención de 64 dimensiones cada una), acompañadas de `Dropout`.
   * **Conexiones residuales y normalización:** Se encuentran encapsuladas en `BertSelfOutput` (después de la atención) y `BertOutput` (después de la red Feed-Forward `BertIntermediate`). Ambas subcapas aplican una proyección `Linear` (`dense`), regularización `Dropout`, suma de la conexión residual (entrada original + salida de la subcapa) y normalización mediante `LayerNorm((768,), eps=1e-12)`.

3. **Dimensión oculta (`hidden_size` o $d_{\text{model}}$):**
   * La dimensión oculta configurada en todas las capas lineales de proyección (`query`, `key`, `value`, `out_features` de salida y `LayerNorm`) es de **768** (`in_features=768, out_features=768`), mientras que la expansión intermedia en la red Feed-Forward (`BertIntermediate`) sube a `3072` ($4 \times 768$).

## 2) Visualización de las Capas y Hidden States 🔍

In [8]:
from transformers import BertTokenizer
import torch

Cargar tokenizer y modelo BERT en español

In [9]:
tokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", output_hidden_states=True)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Texto de ejemplo

In [10]:
text = "El producto de Samsung Galaxy S21 llegó el 12 de marzo y superó mis expectativas."
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
outputs = model(**inputs)

In [11]:
print(outputs)

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.3688, -0.2218, -0.6865,  ..., -0.4430,  0.2883,  0.1794],
         [-0.2616, -0.0662, -0.6243,  ..., -0.5904,  0.6507,  0.2768],
         [-0.2415, -0.2261, -0.6785,  ..., -0.1096,  0.0910,  0.5395],
         ...,
         [-1.4393, -0.1986, -0.5770,  ..., -0.5261,  0.0716,  1.0376],
         [-0.7026,  0.3509, -0.4953,  ..., -0.3322,  0.3734,  0.1284],
         [-1.2273, -0.4597, -1.0296,  ..., -0.6864,  0.9237, -0.5841]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[ 1.0835e-01, -9.4914e-02, -2.0217e-01,  2.8475e-01, -1.8245e-01,
         -6.3078e-01,  1.7496e-01,  2.4555e-01,  3.5131e-01, -5.7070e-01,
         -6.8339e-01, -3.9343e-01, -1.9418e-01, -6.3057e-01, -3.0956e-01,
         -1.5676e-01, -8.3480e-02, -6.0410e-02,  2.2416e-01, -5.0640e-02,
         -1.1274e-01,  3.5219e-01,  8.1201e-02, -4.4294e-01, -9.1450e-02,
         -4.6935e-01,  3.6752e-02,  4.5443e-01, -1.2136e-02,  1.293

Extraer los hidden states


In [12]:
hidden_states = outputs.hidden_states
print("Número de capas (incluyendo la capa de embedding):", len(hidden_states))

Número de capas (incluyendo la capa de embedding): 13


Por ejemplo, si BERT base tiene 12 capas, se imprimirá 13 (1 de embedding + 12 codificadores)

In [13]:
print("Claves de inputs:", inputs.keys())
print("input_ids:", inputs["input_ids"])
print("token_type_ids:", inputs["token_type_ids"])
print("attention_mask:", inputs["attention_mask"])
print("Shape de hidden_states[0]:", hidden_states[0].shape)

Claves de inputs: KeysView({'input_ids': tensor([[    4,  1162,  4196,  1008,  4262,  8643, 16685, 21291, 24010,  1057,
          8926,  4444,  1040,  1797,  1008,  3008,  1042, 27994,  1354, 12956,
          1009,     5]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])})
input_ids: tensor([[    4,  1162,  4196,  1008,  4262,  8643, 16685, 21291, 24010,  1057,
          8926,  4444,  1040,  1797,  1008,  3008,  1042, 27994,  1354, 12956,
          1009,     5]])
token_type_ids: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
Shape de hidden_states[0]: torch.Size([1, 22, 768])


### Análisis Técnico — Sección 2

1. **Inspección del diccionario `inputs` generado por `BertTokenizer`:**
   * **`input_ids`:** Contiene la secuencia de 22 enteros que representan el índice exacto de cada subpalabra (token) dentro del vocabulario de BETO, incluyendo los tokens especiales de inicio `[CLS]` (ID `4`) y cierre `[SEP]` (ID `5`).
   * **`token_type_ids`:** Vector de 22 ceros (`0`) que indica a qué segmento pertenece cada token. En tareas de una sola oración (como este ejemplo), todos los elementos valen `0`; en tareas de pares de oraciones (ej. pregunta-respuesta), la segunda oración tomaría el valor `1`.
   * **`attention_mask`:** Está compuesto exclusivamente por 22 valores `1` porque todos los tokens de la secuencia son contenido real que el modelo debe procesar. Si se hubiera aplicado relleno (`padding`) hasta alcanzar `max_length=128`, las posiciones vacías tendrían valor `0` para que el mecanismo de atención las ignore.

2. **¿Por qué `len(hidden_states)` devuelve `13` en lugar de `12`?**
   * El índice `hidden_states[0]` corresponde a la **salida inicial de la capa de Embeddings** (suma de *word*, *position* y *token_type embeddings* ya normalizados), antes de entrar al primer bloque Transformer. Los índices siguientes (`hidden_states[1]` hasta `hidden_states[12]`) corresponden a las salidas de cada una de las **12 capas codificadoras (`BertLayer`)**.

3. **Significado de las dimensiones de `hidden_states[0].shape` (`torch.Size([1, 22, 768])`):**
   * **Dimensión 0 (`1` — `batch_size`):** Cantidad de textos u oraciones procesadas simultáneamente en el lote (en este caso, 1 sola oración).
   * **Dimensión 1 (`22` — `sequence_length`):** Longitud total de tokens generados tras dividir la oración en subpalabras e incluir `[CLS]` y `[SEP]`.
   * **Dimensión 2 (`768` — `hidden_size`):** Tamaño del vector denso (embedding o representación oculta) con el que BERT representa cada token individual.

## 3) Diagrama Conceptual y Pseudocódigo 🙌

Pseudocódigo del mecanismo de Self-Attention

### Pseudocódigo Paso a Paso del Mecanismo de Self-Attention

Dada la matriz de entrada $X \in \mathbb{R}^{n \times d_{\text{model}}}$ (donde $n$ es la longitud de la secuencia y $d_{\text{model}} = 768$) y las matrices de pesos aprendibles $W^Q, W^K, W^V \in \mathbb{R}^{d_{\text{model}} \times d_k}$:

```text
ALGORITMO Scaled_Dot_Product_Self_Attention(X, W_Q, W_K, W_V, mask):
    # 1. Proyección lineal de los tokens de entrada hacia los espacios Q, K y V
    Q <- X * W_Q        # Queries: lo que cada token busca en los demás
    K <- X * W_K        # Keys: lo que cada token ofrece como identificador
    V <- X * W_V        # Values: el contenido semántico que cada token aporta

    # 2. Cálculo de similitud (producto punto) entre cada Query y todas las Keys
    scores <- Q * Transpuesta(K)    # Matriz de afinidad de dimensión (n x n)

    # 3. Escalamiento para estabilizar los gradientes durante el entrenamiento
    d_k <- dimension_columna(K)     # En BETO: 768 / 12 cabezas = 64
    scaled_scores <- scores / raiz_cuadrada(d_k)

    # 4. Aplicación de máscara de atención (si existe padding)
    SI mask != NULO ENTONCES
        scaled_scores <- reemplazar_donde(mask == 0, valor = -infinito)
    FIN SI

    # 5. Normalización de probabilidades mediante Softmax por filas
    attention_weights <- Softmax(scaled_scores)   # Cada fila suma 1.0

    # 6. Suma ponderada de los valores V según los pesos de atención calculados
    output <- attention_weights * V               # Matriz contextualizada (n x d_k)

    RETORNAR output, attention_weights
FIN ALGORITMO


### Relación Conceptual con la Arquitectura Observada en la Sección 1

1. **¿Por qué se ubica al cierre del laboratorio?**
   * Este bloque funciona como síntesis teórica y algorítmica: permite conectar la estructura estática impresa en la **Sección 1** (`print(model)`) y el flujo dimensional de los tensores inspeccionados en la **Sección 2** (`[batch_size, sequence_length, hidden_size]`) con la operación matemática exacta que ocurre dentro de cada capa oculta para contextualizar las palabras.

2. **Vínculo directo con `BertSelfAttention`:**
   * En la salida de `print(model)` (Sección 1), cada uno de los 12 bloques `BertLayer` contiene el submódulo `attention.self` (`BertSelfAttention`) con tres capas `Linear(in_features=768, out_features=768)` llamadas `query`, `key` y `value`.
   * Esas tres capas lineales implementan físicamente el **Paso 1** del pseudocódigo ($X W^Q$, $X W^K$, $X W^V$), pero ejecutándolo en paralelo a través de **12 cabezas de atención** (*Multi-Head Attention*, donde cada cabeza opera con $d_k = 768 / 12 = 64$). Finalmente, las salidas de las 12 cabezas se concatenan y pasan por `attention.output.dense` para recuperar la dimensión `768` antes de sumar la conexión residual y aplicar `LayerNorm`.